# 6. MiniLM + FAISS with 0.55 cutoff

Pure semantic control with calibrated abstention; retrieval below improved BM25.

This is a recorded full-train comparison, not a new evaluation.

In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
from presentation_review import selection,cached_demo
model=selection(6)
model['config_snapshot']

{'name': 'dense_minilm_abstention',
 'retriever_type': 'dense',
 'model_id': 'sentence-transformers/all-MiniLM-L6-v2',
 'revision': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41',
 'dimension': 384,
 'max_seq_length': 256,
 'device': 'cpu',
 'torch_threads': 4,
 'batch_size': 32,
 'chunk_size': 1000,
 'chunk_overlap': 200,
 'min_score': 0.0,
 'k_values': [1, 3, 5, 10],
 'query_mode': 'full',
 'normalize_embeddings': True,
 'faiss_index': 'IndexFlatIP',
 'abstention_threshold': 0.55}

In [2]:
m=model['train_metrics']
table=pd.DataFrame(m['retrieval'])[['k','n_queries','precision','recall','hit']]
table

,k,n_queries,precision,recall,hit
0,1,140,0.052693,0.108877,0.157143
1,3,140,0.044502,0.257516,0.357143
2,5,140,0.040341,0.357489,0.464286
3,10,140,0.028280,0.482709,0.585714


In [3]:
print('Correct negative abstention:',m['negative_counts'])
print('False answerable abstention:',m['false_abstention'])
print('Cached latency:',m['latency_ms'])

Correct negative abstention: {'n': 23, 'correct': 9}
False answerable abstention: 0.007142857142857143
Cached latency: {'median': 1.4008500147610903, 'p95': 1.7931099049746981}


## Cached example

The question comes from recorded train predictions. Config settings and original source offsets remain visible; the model cannot call a hosted provider here.

In [4]:
demo=cached_demo(model)
print(demo['query'])
print(demo.get('summary') or demo['message'])
pd.DataFrame(demo['hits'])[['file_path','start','end','text']]

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?
5 passages retrieved.


,file_path,start,end,text
0,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,cuad/HOLIDAYRVSUPERSTORESINC_04_15_2002-EX-10....,4800,5800,"he License (the ""Endorsement"").\n\n 3. ..."
2,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."
3,"cuad/Loop Industries, Inc. - Marketing Agreeme...",19200,20200,"greement; or (vii) in confidence, in connectio..."
4,"cuad/DUOSTECHNOLOGIESGROUP,INC_04_21_2009-EX-1...",18400,19400,enses expressly granted under this Agreement. ...


Full code walkthrough: ../experiments/17_hybrid_bm25_minilm.ipynb. Historical acceptance and current preference are separate; see REPORT.md for limitations.